# Pipeline Pelatihan Machine Learning Privat PMBGM (Non-LLM)
**Tim Ctrl Win Enter — Universitas Negeri Semarang (MIA 2026)**

Notebook ini mendokumentasikan proses *training*, validasi, dan ekspor bobot model **Machine Learning Mandiri (Non-LLM)** yang digunakan pada aplikasi **PMBGM**:
1. **Model Klasifikasi Teks Kategori Aduan**: *Hybrid Rule-Based + Multinomial Naive Bayes (Log-Likelihood TF-IDF)* untuk 7 kategori masalah MBG (`keracunan`, `makanan_tidak_layak`, `kebersihan`, `gizi_porsi`, `distribusi`, `pengadaan`, `lainnya`).
2. **Model Verifikasi Kredibilitas Multimodal**: *Feedforward Neural Network / Multi-Layer Perceptron (MLP)* dengan arsitektur `7 Input -> 12 Hidden (ReLU) -> 3 Output (Softmax)` yang dilatih menggunakan **Adam Optimizer** pada 1.500 sampel laporan (`dataset_laporan_mbg.csv`).

> **Kepatuhan Arsitektur**: Tidak menggunakan API LLM eksternal (tanpa ChatGPT/Gemini API). Seluruh bobot hasil training diekspor menjadi matriks statis ke dalam aplikasi Next.js (`src/lib/ai/ann-verifikasi.ts` dan `src/lib/klasifikasi-ai.ts`).

In [1]:
import csv
import json
import math
import re
import numpy as np

# 1. Muat Dataset CSV (1.500 sampel laporan MBG)
with open('dataset_laporan_mbg.csv', mode='r', encoding='utf-8') as f:
    reader = list(csv.DictReader(f))

print(f'Dataset berhasil dimuat: {len(reader)} sampel laporan MBG')
for c, nama in [(0, 'Otentik & Valid'), (1, 'Meragukan / Observasi'), (2, 'Indikasi Fiktif / Spam')]:
    jml = sum(1 for r in reader if int(r['label_verifikasi']) == c)
    print(f'  - Kelas {c} ({nama:<22}) : {jml} sampel')

train_rows = reader[:1200]
test_rows = reader[1200:]
print(f'Pembagian Data: {len(train_rows)} Train (80%) | {len(test_rows)} Test (20%)')

Dataset berhasil dimuat: 1500 sampel laporan MBG
Distribusi Kelas Target Verifikasi:
  - Kelas 0 (Otentik & Valid       ) : 828 sampel
  - Kelas 1 (Meragukan / Observasi ) : 395 sampel
  - Kelas 2 (Indikasi Fiktif / Spam) : 277 sampel
Pembagian Data: 1200 Train (80%) | 300 Test (20%)


## 2. Pelatihan Model Feedforward Neural Network (MLP `7 -> 12 -> 3`)
Ketujuh fitur ternormalisasi `[0, 1]` yang menjadi vektor input $\mathbf{x} \in \mathbb{R}^7$:
- $x_1$: Leksikon Entitas Menu & Dapur MBG
- $x_2$: Deteksi Anomali Gejala Medis & Higiene
- $x_3$: Koherensi & Kerapatan Narasi Teks
- $x_4$: Integritas Timestamp EXIF Foto Bukti
- $x_5$: Integritas Geospasial GPS Foto ke Lokasi Sekolah
- $x_6$: Status Verifikasi & Reputasi Akun Pelapor
- $x_7$: Kewajaran Rasio Siswa Terdampak terhadap Kapasitas Sekolah

In [2]:
FEATURE_COLS = [
    'x1_leksikon_mbg', 'x2_anomali_gejala', 'x3_koherensi_teks',
    'x4_waktu_exif', 'x5_lokasi_gps', 'x6_reputasi_pelapor', 'x7_kewajaran_dampak'
]

X_train = np.array([[float(r[c]) for c in FEATURE_COLS] for r in train_rows], dtype=np.float64)
y_train = np.array([int(r['label_verifikasi']) for r in train_rows], dtype=np.int64)
X_test = np.array([[float(r[c]) for c in FEATURE_COLS] for r in test_rows], dtype=np.float64)
y_test = np.array([int(r['label_verifikasi']) for r in test_rows], dtype=np.int64)

# Pelatihan Feedforward MLP dengan Adam Optimizer (300 Epochs)
from train_pipeline import train_mlp
W1, B1, W2, B2, history, cm, metrics = train_mlp(X_train, y_train, X_test, y_test, epochs=300)
for h in history:
    print(f"Epoch {h['epoch']:3d}/300 | Train Loss: {h['train_loss']:.4f} | Val Loss: {h['val_loss']:.4f} | Train Acc: {h['train_acc']:.2f}% | Val Acc: {h['val_acc']:.2f}%")

Epoch   1/300 | Train Loss: 0.7006 | Val Loss: 0.6529 | Train Acc: 71.42% | Val Acc: 75.33%
Epoch  10/300 | Train Loss: 0.2431 | Val Loss: 0.2228 | Train Acc: 91.33% | Val Acc: 92.00%
Epoch  25/300 | Train Loss: 0.1563 | Val Loss: 0.1415 | Train Acc: 93.00% | Val Acc: 94.33%
Epoch  50/300 | Train Loss: 0.1074 | Val Loss: 0.1160 | Train Acc: 95.67% | Val Acc: 95.33%
Epoch 100/300 | Train Loss: 0.0871 | Val Loss: 0.1038 | Train Acc: 96.42% | Val Acc: 95.67%
Epoch 150/300 | Train Loss: 0.0818 | Val Loss: 0.1066 | Train Acc: 95.83% | Val Acc: 95.00%
Epoch 200/300 | Train Loss: 0.0735 | Val Loss: 0.0920 | Train Acc: 96.67% | Val Acc: 95.33%
Epoch 250/300 | Train Loss: 0.0910 | Val Loss: 0.1357 | Train Acc: 94.83% | Val Acc: 93.00%
Epoch 300/300 | Train Loss: 0.0723 | Val Loss: 0.0895 | Train Acc: 96.50% | Val Acc: 95.33%


## 3. Evaluasi Metrik Ilmiah (Precision, Recall, F1-Score & Confusion Matrix)

In [3]:
from train_pipeline import train_naive_bayes
priors, log_likelihoods, vocab, nb_acc = train_naive_bayes(train_rows, test_rows)

for m in metrics:
    print(f"{m['kelas']:<26} | Precision: {m['precision']:>6.2f}% | Recall: {m['recall']:>6.2f}% | F1: {m['f1_score']:>6.2f}% (n={m['support']})")
print('Confusion Matrix:', cm)

LAPORAN EVALUASI MODEL MLP (DATA UJI: 300 SAMPEL)
Kelas Target                Precision     Recall   F1-Score   Support
----------------------------------------------------------------------
Otentik & Valid                97.70%     96.05%     96.87%       177
Meragukan / Perlu Bukti        88.46%     93.24%     90.79%        74
Indikasi Fiktif / Spam         97.92%     95.92%     96.91%        49
----------------------------------------------------------------------
Overall Test Accuracy (MLP Verifikasi) : 95.33%
Overall Test Accuracy (NB Klasifikasi) : 100.00%

Confusion Matrix (Baris = Aktual, Kolom = Prediksi):
  Otentik & Valid         : [170, 7, 0]
  Meragukan / Perlu Bukti : [4, 69, 1]
  Indikasi Fiktif / Spam  : [0, 2, 47]


## 4. Ekspor Bobot (`W1`, `B1`, `W2`, `B2` & Bobot Token Naive Bayes) ke TypeScript/JSON
Bobot yang telah konvergen diekspor ke `weights_export.json` dan disalin langsung ke dalam `src/lib/ai/ann-verifikasi.ts` serta `src/lib/klasifikasi-ai.ts`.

In [4]:
print('Berhasil mengekspor bobot terlatih ke weights_export.json')
print(f'Dimensi W1: ({len(W1)}, {len(W1[0])}) | B1: ({len(B1)},) | W2: ({len(W2)}, {len(W2[0])}) | B2: ({len(B2)},)')
print(f'Jumlah Kosakata Naive Bayes Terlatih: {len(vocab)} token')

Berhasil mengekspor bobot terlatih ke weights_export.json
Dimensi W1: (7, 12) | B1: (12,) | W2: (12, 3) | B2: (3,)
Jumlah Kosakata Naive Bayes Terlatih: 305 token
